# Tubes IPSD - Klasifikasi MonkeyPox Skin Lesion (5-Fold Cross Validation)
Notebook ini didesain self-contained agar dapat langsung dijalankan, diedit, dan dioptimasi (mengatasi overfitting) di **Google Colab**.

* **Model:** `regnet_y_800mf` (Meta AI, Squeeze-and-Excitation, AnyNet space)
* **Dataset:** Mpox Skin Lesion Dataset v2.0 (Original Train + Augmented Train -> Train; Original Valid -> Valid; Original Test -> Test)
* **Metrik Evaluasi:** Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix (Rerata 5-Fold CV)



In [ ]:
# 1. Cek Device & Dependensi
import torch
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nama GPU        :", torch.cuda.get_device_name(0))
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

!pip install -q pandas scikit-learn matplotlib seaborn



In [ ]:
# 2. Download & Ekstrak Dataset via Kaggle API
import os
import json

# Setup Kredensial Kaggle
os.environ['KAGGLE_USERNAME'] = "dhitanhakim"
os.environ['KAGGLE_KEY'] = "KGAT_ce2cf0ac4ed0304ec0d562e3aafda388"

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({"username": "dhitanhakim", "key": "KGAT_ce2cf0ac4ed0304ec0d562e3aafda388"}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

# Download & Unzip dataset
!kaggle datasets download -d joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20
!unzip -q -o mpox-skin-lesion-dataset-version-20-msld-v20.zip -d Data
print("Dataset berhasil diekstrak ke folder 'Data'!")



In [ ]:
# 3. Konfigurasi & Hyperparameter (Bisa diedit langsung untuk atasi overfitting)
class Config:
    model_name = "regnet_y_800mf"
    num_classes = 6
    img_size = 224
    batch_size = 16
    epochs = 15
    lr = 5e-5
    
    # --- PENGATURAN ANTI-OVERFITTING ---
    weight_decay = 1e-2      # L2 Regularization lebih agresif mencegah overfitting
    dropout_rate = 0.4       # Dropout sebelum layer klasifikasi
    label_smoothing = 0.10   # Mencegah overconfidence probabilitas
    patience = 5             # Early stopping patience (berhenti jika val loss tidak membaik)
    
    # Path & Device
    data_dir = "Data"
    results_dir = "Results"
    checkpoints_dir = "checkpoints"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(Config.results_dir, exist_ok=True)
os.makedirs(Config.checkpoints_dir, exist_ok=True)
print("Config siap. Menggunakan device:", Config.device)



In [ ]:
# 4. Pipeline Augmentasi Data & DataLoader
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_transforms(img_size=224):
    # Augmentasi agresif pada data Train untuk mereduksi overfitting
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return train_transform, eval_transform

def find_fold_dir(base_dir, fold, split, aug=False):
    split_norm = split.strip().lower()
    fold_marker = f"fold{fold}"

    for root, dirs, _ in os.walk(base_dir):
        root_low = root.lower().replace(" ", "").replace("_", "")
        base_name_low = os.path.basename(root).strip().lower()

        is_correct_split = (base_name_low == split_norm)
        has_fold = fold_marker in root_low
        is_aug_match = ("aug" in root_low) if aug else ("aug" not in root_low)

        if is_correct_split and has_fold and is_aug_match and dirs:
            return root
    raise FileNotFoundError(f"Folder '{split}' fold {fold} (aug={aug}) tidak ditemukan di {base_dir}")

def get_fold_loaders(data_dir, fold, batch_size=16, img_size=224):
    train_tf, eval_tf = get_transforms(img_size=img_size)

    orig_train_dir = find_fold_dir(data_dir, fold=fold, split="Train", aug=False)
    aug_train_dir  = find_fold_dir(data_dir, fold=fold, split="Train", aug=True)
    valid_dir      = find_fold_dir(data_dir, fold=fold, split="Valid", aug=False)
    test_dir       = find_fold_dir(data_dir, fold=fold, split="Test", aug=False)

    orig_train_ds = datasets.ImageFolder(orig_train_dir, transform=train_tf)
    aug_train_ds  = datasets.ImageFolder(aug_train_dir, transform=train_tf)
    train_dataset = ConcatDataset([orig_train_ds, aug_train_ds])

    valid_dataset = datasets.ImageFolder(valid_dir, transform=eval_tf)
    test_dataset  = datasets.ImageFolder(test_dir, transform=eval_tf)

    class_names = orig_train_ds.classes
    use_pin = torch.cuda.is_available()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=use_pin, num_workers=2)
    val_loader   = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, pin_memory=use_pin, num_workers=2)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=use_pin, num_workers=2)

    return train_loader, val_loader, test_loader, class_names



In [ ]:
# 5. Arsitektur Model (RegNet-Y-800MF dengan Dropout)
import torch.nn as nn
from torchvision import models

def build_model(model_name=Config.model_name, num_classes=Config.num_classes, dropout_rate=Config.dropout_rate, pretrained=True):
    if "regnet" in model_name.lower():
        weights = models.RegNet_Y_800MF_Weights.DEFAULT if pretrained else None
        model = models.regnet_y_800mf(weights=weights)
        in_features = model.fc.in_features
        
        # Tambahkan Dropout sebelum layer linear akhir untuk mencegah overfitting
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, num_classes)
        )
        return model
    else:
        raise ValueError(f"Model {model_name} belum didukung.")



In [ ]:
# 6. Fungsi Evaluasi Metrik & Visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

def compute_metrics(y_true, y_pred, y_prob, num_classes=6):
    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_macro  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)

    try:
        roc_auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        roc_auc = float("nan")

    cm = confusion_matrix(y_true, y_pred)

    return {
        "accuracy": float(acc),
        "precision_macro": float(prec_macro),
        "recall_macro": float(rec_macro),
        "f1_macro": float(f1_macro),
        "roc_auc_macro": float(roc_auc),
        "confusion_matrix": cm,
    }

def plot_learning_curves(history, fold, save_dir=Config.results_dir):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(epochs, history["train_loss"], label="Train Loss", color="#d9534f", lw=2)
    axes[0].plot(epochs, history["valid_loss"], label="Valid Loss", color="#0275d8", lw=2)
    axes[0].set_title(f"Loss Curve - Fold {fold}")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.4)

    axes[1].plot(epochs, history["train_acc"], label="Train Acc", color="#5cb85c", lw=2)
    axes[1].plot(epochs, history["valid_acc"], label="Valid Acc", color="#f0ad4e", lw=2)
    axes[1].set_title(f"Accuracy Curve - Fold {fold}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"learning_curve_fold{fold}.png"), dpi=200)
    plt.show()

def plot_confusion_matrix(cm, class_names, fold, save_dir=Config.results_dir):
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix (Test Set) - Fold {fold}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"confusion_matrix_fold{fold}.png"), dpi=200)
    plt.show()

def plot_roc_curves(y_true, y_prob, class_names, fold, save_dir=Config.results_dir):
    y_prob = np.asarray(y_prob)
    num_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
    if num_classes == 2:
        y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])

    plt.figure(figsize=(8, 6))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        plt.plot(fpr, tpr, lw=1.5, label=f"{class_names[i]} (AUC={auc(fpr, tpr):.3f})")

    plt.plot([0, 1], [0, 1], "k--", lw=1.5, label="Chance")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (Test Set) - Fold {fold}")
    plt.legend(loc="lower right"); plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"roc_curve_fold{fold}.png"), dpi=200)
    plt.show()



In [ ]:
# 7. Engine Training, Early Stopping & Testing
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = (scaler is not None and device.type == "cuda")

    for images, targets in loader:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0
    all_targets, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, targets)

            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            total_loss += loss.item() * images.size(0)
            total += targets.size(0)

            all_targets.extend(targets.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())

    return total_loss / total, all_targets, all_preds, all_probs

def run_single_fold(fold, cfg=Config):
    print(f"\n{'='*25} MULAI FOLD {fold} / 5 {'='*25}")
    train_loader, val_loader, test_loader, class_names = get_fold_loaders(
        cfg.data_dir, fold, batch_size=cfg.batch_size, img_size=cfg.img_size
    )

    model = build_model(cfg.model_name, cfg.num_classes, dropout_rate=cfg.dropout_rate).to(cfg.device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)

    scaler = torch.amp.GradScaler("cuda") if cfg.device.type == "cuda" else None
    best_val_loss = float("inf")
    best_ckpt_path = os.path.join(cfg.checkpoints_dir, f"best_fold{fold}.pt")
    history = {"train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": []}

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, cfg.device, scaler)
        val_loss, val_targets, val_preds, _ = evaluate(model, val_loader, criterion, cfg.device)
        scheduler.step()

        val_acc = (torch.tensor(val_preds) == torch.tensor(val_targets)).float().mean().item()
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["valid_loss"].append(val_loss)
        history["valid_acc"].append(val_acc)

        print(f"Fold {fold} | Epoch {epoch:02d}/{cfg.epochs:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # Simpan checkpoint terbaik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_ckpt_path)

    # Plot kurva loss & acc
    plot_learning_curves(history, fold=fold)

    # Evaluasi pada TEST SET
    print(f"\n[Fold {fold}] Menguji model terbaik pada TEST SET...")
    model.load_state_dict(torch.load(best_ckpt_path, map_location=cfg.device))
    _, test_targets, test_preds, test_probs = evaluate(model, test_loader, criterion, cfg.device)

    metrics = compute_metrics(test_targets, test_preds, test_probs, cfg.num_classes)
    metrics["fold"] = fold
    print(f"Hasil Test Fold {fold} -> Acc: {metrics['accuracy']:.4f} | Prec: {metrics['precision_macro']:.4f} | Rec: {metrics['recall_macro']:.4f} | F1: {metrics['f1_macro']:.4f} | ROC-AUC: {metrics['roc_auc_macro']:.4f}")

    plot_confusion_matrix(metrics["confusion_matrix"], class_names, fold=fold)
    plot_roc_curves(test_targets, test_probs, class_names, fold=fold)

    return metrics



In [ ]:
# 8. Eksekusi 5-Fold Cross Validation Lengkap
all_fold_metrics = []

for fold in range(1, 6):
    fold_result = run_single_fold(fold, Config)
    all_fold_metrics.append(fold_result)

# Buat Tabel Ringkasan Akhir
records = []
for item in all_fold_metrics:
    records.append({
        "Fold": f"Fold {item['fold']}",
        "Accuracy": item["accuracy"],
        "Precision (Macro)": item["precision_macro"],
        "Recall (Macro)": item["recall_macro"],
        "F1-Score (Macro)": item["f1_macro"],
        "ROC-AUC (Macro)": item["roc_auc_macro"]
    })

df_summary = pd.DataFrame(records)
mean_vals = df_summary.mean(numeric_only=True).to_dict()
std_vals  = df_summary.std(numeric_only=True).to_dict()
mean_vals["Fold"] = "Mean (Rerata)"
std_vals["Fold"]  = "Std Dev"

df_final = pd.concat([df_summary, pd.DataFrame([mean_vals, std_vals])], ignore_index=True)
df_final.to_csv(os.path.join(Config.results_dir, "summary_5fold_metrics.csv"), index=False)

print("\n" + "="*60)
print("HASIL LENGKAP 5-FOLD CROSS VALIDATION (TEST SET):")
print("="*60)
display(df_final)



In [ ]:
# 9. Download Semua Gambar Hasil Evaluasi & Tabel CSV
!zip -r mpox_results.zip Results
from google.colab import files
files.download('mpox_results.zip')
print("File 'mpox_results.zip' siap diunduh dan di-upload ke GitHub repo!")

